# Разделение большого лога на части

Загрузи этот ноутбук, укажи путь к файлу и размер части — он разобьёт JSONL на несколько файлов `.xml` (как ты уже присылал), каждый влезающий в лимит загрузки.

**Лимит чата: 25 МБ.** Рекомендую размер части = **20 МБ** (с запасом).

In [13]:
# === НАСТРОЙКА ===
# Путь к исходному файлу (JSONL или .xml с JSON-строками)
INPUT_FILE = r"C:\Users\Administrator\Documents\Программы\Poly\Polybot\polymarket_bot\logs\demo_interval_samples.jsonl"

# Размер одной части в мегабайтах (25 МБ лимит чата → ставь 20)
PART_SIZE_MB = 20

# Папка для частей (по умолчанию — рядом с исходным файлом)
# Можно указать отдельную, например: r"C:\Users\Administrator\Desktop\parts"
OUTPUT_DIR = None  # None = рядом с файлом

# Расширение выходных файлов
EXT = ".xml"

import os
print(f"Исходный файл: {INPUT_FILE}")
print(f"Размер части: {PART_SIZE_MB} МБ")
print(f"Папка вывода: {OUTPUT_DIR or os.path.dirname(INPUT_FILE)}")

Исходный файл: C:\Users\Administrator\Documents\Программы\Poly\Polybot\polymarket_bot\logs\demo_interval_samples.jsonl
Размер части: 20 МБ
Папка вывода: C:\Users\Administrator\Documents\Программы\Poly\Polybot\polymarket_bot\logs


In [14]:
import os, math

if OUTPUT_DIR is None:
    OUTPUT_DIR = os.path.dirname(INPUT_FILE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

basename = os.path.splitext(os.path.basename(INPUT_FILE))[0]
part_size = PART_SIZE_MB * 1024 * 1024  # bytes

part_num = 1
cur_size = 0
cur_lines = 0
cur_file = None

total_lines = 0
total_size = os.path.getsize(INPUT_FILE)
print(f"Исходный файл: {total_size/1024/1024:.1f} МБ")
print(f"Будет разбит на ~{math.ceil(total_size/part_size)} частей по {PART_SIZE_MB} МБ")
print("---")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line_bytes = line.encode("utf-8")
        line_len = len(line_bytes)

        # открываем новый файл если нужно
        if cur_file is None or cur_size + line_len > part_size:
            if cur_file:
                cur_file.close()
                print(f"  часть {part_num-1}: {cur_size/1024/1024:.1f} МБ, {cur_lines} строк → {os.path.basename(out_path)}")
            out_path = os.path.join(OUTPUT_DIR, f"{basename}_part{part_num:02d}{EXT}")
            cur_file = open(out_path, "w", encoding="utf-8")
            cur_size = 0
            cur_lines = 0
            part_num += 1

        cur_file.write(line)
        cur_size += line_len
        cur_lines += 1
        total_lines += 1

if cur_file:
    cur_file.close()
    print(f"  часть {part_num-1}: {cur_size/1024/1024:.1f} МБ, {cur_lines} строк → {os.path.basename(out_path)}")

print(f"\nГОТОВО: {total_lines} строк → {part_num-1} файлов в {OUTPUT_DIR}")
print("Файлы готовы к отправке в чат по одному.")

Исходный файл: 127.9 МБ
Будет разбит на ~7 частей по 20 МБ
---
  часть 1: 20.0 МБ, 99777 строк → demo_interval_samples_part01.xml
  часть 2: 20.0 МБ, 100256 строк → demo_interval_samples_part02.xml
  часть 3: 20.0 МБ, 100130 строк → demo_interval_samples_part03.xml
  часть 4: 20.0 МБ, 99923 строк → demo_interval_samples_part04.xml
  часть 5: 20.0 МБ, 100437 строк → demo_interval_samples_part05.xml
  часть 6: 20.0 МБ, 100018 строк → demo_interval_samples_part06.xml
  часть 7: 7.3 МБ, 35721 строк → demo_interval_samples_part07.xml

ГОТОВО: 636262 строк → 7 файлов в C:\Users\Administrator\Documents\Программы\Poly\Polybot\polymarket_bot\logs
Файлы готовы к отправке в чат по одному.


## Обратная операция (склеивание — для меня)

Когда ты пришлёшь все части, я склею их простой конкатенацией (JSONL строки не зависят от порядка для анализа, но хронологический порядок сохраняется автоматически при разбиении).

In [15]:
# Проверка: размеры частей
files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.startswith(basename + "_part") and f.endswith(EXT)])
print(f"Создано файлов: {len(files)}")
for fname in files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    sz = os.path.getsize(fpath)
    with open(fpath, encoding="utf-8") as f:
        n = sum(1 for _ in f)
    ok = "✓" if sz < 25*1024*1024 else "✗ ПРЕВЫШЕН ЛИМИТ!"
    print(f"  {fname}: {sz/1024/1024:.1f} МБ, {n} строк {ok}")

Создано файлов: 9
  demo_interval_samples_part01.xml: 20.1 МБ, 99777 строк ✓
  demo_interval_samples_part01_24.08.2026.xml: 20.1 МБ, 99777 строк ✓
  demo_interval_samples_part02.xml: 20.1 МБ, 100256 строк ✓
  demo_interval_samples_part02_24.08.2026.xml: 15.2 МБ, 75634 строк ✓
  demo_interval_samples_part03.xml: 20.1 МБ, 100130 строк ✓
  demo_interval_samples_part04.xml: 20.1 МБ, 99923 строк ✓
  demo_interval_samples_part05.xml: 20.1 МБ, 100437 строк ✓
  demo_interval_samples_part06.xml: 20.1 МБ, 100018 строк ✓
  demo_interval_samples_part07.xml: 7.3 МБ, 35721 строк ✓
